# DocRED 데이터 확인 (프로젝트 설계 단계 - 데이터 파악용)

Wikipedia/Wikidata 기반 **문서 단위 관계 추출(Document-level Relation Extraction)** 데이터셋.
한 문서(여러 문장) 안의 엔티티들을 찾고, 그 엔티티 쌍 사이의 관계를 추론하는 태스크다.

각 split이 몇 개의 문서로 이뤄져 있는지, 각 문서의 구조(문장/엔티티/관계)를
head 몇 개만 뽑아서 사람이 읽을 수 있는 형태로 확인한다.
관계 ID(`P17` 등)는 `rel_info.json`으로 사람이 읽는 텍스트(`country` 등)로 변환해서 본다.


## 1. 준비 (경로 / 상수)

In [1]:
import json
import os

DATA_DIR = os.path.join("docred_data", "data")
REL_INFO_PATH = os.path.join(DATA_DIR, "rel_info.json")

# 크기가 작은 순서. train_distant(417MB)는 무거워서 기본 요약에서 제외.
SMALL_SPLITS = ["dev.json", "test.json", "train_annotated.json"]


## 2. 관계 정보(rel_info) 로드
관계 ID → 사람이 읽는 관계명 매핑 (예: `P17` → `country`).

In [2]:
with open(REL_INFO_PATH, encoding="utf-8") as f:
    rel_info = json.load(f)

print(f"관계 종류 수: {len(rel_info)}")
list(rel_info.items())[:5]


관계 종류 수: 96


[('P6', 'head of government'),
 ('P17', 'country'),
 ('P19', 'place of birth'),
 ('P20', 'place of death'),
 ('P22', 'father')]

## 3. 헬퍼 함수
문서 1개를 사람이 읽기 좋은 형태로 요약해서 출력하는 함수들.

In [3]:
def sent_to_text(tokens):
    """토큰 리스트를 대략적인 문장 문자열로 복원 (확인용이라 단순 공백 join)."""
    return " ".join(tokens)


def describe_document(doc, rel_info, max_items=3):
    """문서 1개를 사람이 읽기 좋은 형태로 요약 출력."""
    sents = doc["sents"]
    vertex_set = doc["vertexSet"]
    labels = doc.get("labels", [])  # test split에는 labels가 없음

    print(f"  title        : {doc['title']}")
    print(f"  # sentences  : {len(sents)}")
    print(f"  # entities   : {len(vertex_set)}")
    print(f"  # relations  : {len(labels)}")

    # 1) 첫 문장 몇 개를 텍스트로 복원해서 보여주기
    print("  -- sentences (head) --")
    for i, tokens in enumerate(sents[:max_items]):
        print(f"    [{i}] {sent_to_text(tokens)}")

    # 2) 엔티티 몇 개: 엔티티는 '동일 대상을 가리키는 mention들의 묶음'
    print("  -- entities (head) --")
    for e_idx, mentions in enumerate(vertex_set[:max_items]):
        names = sorted({m["name"] for m in mentions})
        etype = mentions[0]["type"]
        print(f"    E{e_idx} ({etype}): {names}  (mentions={len(mentions)})")

    # 3) 관계 몇 개: head 엔티티 --관계--> tail 엔티티 형태로 디코딩
    if labels:
        print("  -- relations (head) --")
        for lab in labels[:max_items]:
            h_name = vertex_set[lab["h"]][0]["name"]
            t_name = vertex_set[lab["t"]][0]["name"]
            rel_txt = rel_info.get(lab["r"], lab["r"])
            evi = lab.get("evidence", [])
            print(f"    {h_name}  --[{rel_txt} / {lab['r']}]-->  {t_name}   (evidence sents={evi})")
    else:
        print("  -- relations: 없음 (test split은 정답 라벨 비공개) --")


def inspect_file(path, rel_info, n_docs=1):
    """split 파일 1개를 로드해서 전체 개수 + 앞쪽 문서 n개를 요약."""
    print("=" * 70)
    print(f"FILE: {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    print(f"  top-level type : {type(data).__name__}")
    print(f"  # documents    : {len(data)}")
    print(f"  document keys  : {list(data[0].keys())}")
    for i in range(min(n_docs, len(data))):
        print(f"\n  ── document [{i}] ──")
        describe_document(data[i], rel_info)


## 4. 각 split 확인
작은 3개 split(dev / test / train_annotated)을 순서대로 요약한다.

In [4]:
for name in SMALL_SPLITS:
    inspect_file(os.path.join(DATA_DIR, name), rel_info, n_docs=1)


FILE: docred_data\data\dev.json  (4.3 MB)
  top-level type : list
  # documents    : 998
  document keys  : ['vertexSet', 'labels', 'title', 'sents']

  ── document [0] ──
  title        : Skai TV
  # sentences  : 7
  # entities   : 11
  # relations  : 7
  -- sentences (head) --
    [0] Skai TV is a Greek free - to - air television network based in Piraeus .
    [1] It is part of the Skai Group , one of the largest media groups in the country .
    [2] It was relaunched in its present form on 1st of April 2006 in the Athens metropolitan area , and gradually spread its coverage nationwide .
  -- entities (head) --
    E0 (ORG): ['Skai TV']  (mentions=3)
    E1 (LOC): ['Greek']  (mentions=1)
    E2 (LOC): ['Piraeus']  (mentions=1)
  -- relations (head) --
    Piraeus  --[country / P17]-->  Greece   (evidence sents=[0, 4])
    Skai Group  --[country / P17]-->  Greece   (evidence sents=[0, 1, 4])
    Athens  --[country / P17]-->  Greece   (evidence sents=[0, 2, 4])
FILE: docred_data\data\t

## 5. 참고

- **`train_distant.json` (417MB, 원거리 지도학습용)** 은 크기가 커서 위 요약에서 제외했다.
  필요하면 아래 셀에서 확인 (로드에 시간이 걸릴 수 있음).
- 저장소 `README.md`의 필드 형식(`labels`가 병렬 리스트 딕셔너리, `relation_text` 포함)은
  **HuggingFace 변환본** 형식이고, 실제 로컬 JSON은 **원본 DocRED 형식**(`labels`가 dict의 리스트,
  관계 텍스트는 `rel_info.json`으로 별도 매핑)이다. 로더는 실제 파일 형식 기준으로 작성해야 한다.


In [5]:
# inspect_file(os.path.join(DATA_DIR, "train_distant.json"), rel_info, n_docs=1)
